# Assemble + publish chessbench-full

Downloads the 8 shard pieces from HF, merges them into `train_set.npz` + `teacher_logp.npy` (runbook contract), and publishes the Kaggle Dataset `chessbench-full` (public, so all three accounts can mount it). Run once after all shards are built.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
except Exception as exc:
    print('HF secret unavailable:', exc)
assert os.environ.get('HF_WRITE_TOKEN'), 'create Kaggle secret HF_WRITE_TOKEN with write access to vedangfake/chess-slm-benchmark'
os.chdir(REPO)

In [ ]:
# Self-wait: poll HF until all 8 shards are built (builders run in parallel
# across accounts; this CPU kernel waits and assembles automatically).
# NOTE: manifest is advisory and overwritten by parallel builders — use file list as source of truth.
import time
from huggingface_hub import HfApi
api = HfApi(token=os.environ.get('HF_WRITE_TOKEN'))
while True:
    try:
        files = set(api.list_repo_files('vedangfake/chess-slm-benchmark', repo_type='dataset'))
        n_done = sum(1 for i in range(8) if f'chessbench-full-build/shard-{i:05d}/train_set.npz' in files and f'chessbench-full-build/shard-{i:05d}/teacher_logp.npy' in files)
        print(f'[wait] shards ready: {n_done}/8', flush=True)
        if n_done >= 8:
            break
    except Exception as exc:
        print(f'[wait] check failed: {exc}', flush=True)
    time.sleep(60)
cmd = [sys.executable, 'scripts/assemble_full_dataset.py',
       '--n-shards', '8',
       '--hf-repo', 'vedangfake/chess-slm-benchmark',
       '--hf-run', 'chessbench-full-build',
       '--out', '/kaggle/working/chessbench-full']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('ASSEMBLED')

In [ ]:
# Publish as a public Kaggle Dataset so all three accounts can mount it.
import json
from pathlib import Path
out = Path('/kaggle/working/chessbench-full')
meta = {
    'id': 'ACCOUNT/chessbench-full',
    'title': 'chessbench-full',
    'subtitle': 'ChessBench train action-value slice: tokens/actions/winprob + 9M teacher log-probs [8 shards]',
    'isPrivate': False,
    'licenses': [{'name': 'other'}]}
(out / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
import os
os.environ['KAGGLE_USERNAME'] = 'ACCOUNT'
r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(out)],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
print(r.stderr[-2000:])
print('DATASET CREATED' if r.returncode == 0 else 'PUBLISH FAILED - see output')